In [ ]:
# ======================================================
# Notebook 02: Metadata Integration
#
# Goal:
# Audit and integrate patient-level clinical metadata
# with the integrated breast cancer single-cell atlas.
#
# Output:
# Master patient metadata table for downstream analyses.
# ======================================================

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

adata = sc.read_h5ad(
    "data/IntegratedAtlas.h5ad",
    backed="r"
)

print(adata)

AnnData object with n_obs × n_vars = 621200 × 37389 backed at 'data\\IntegratedAtlas.h5ad'
    obs: 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'grade', 'author_cell_type', 'batch', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_rpca', 'X_umap'


In [2]:
import os

os.listdir("data/integratedatlas_supplemental_files")

['SuppTable1.xlsx',
 'SuppTable2 (epi).xlsx',
 'SuppTable3 (imm).xlsx',
 'SuppTable4 (strom).xlsx',
 'supp_figures.pdf']

In [3]:
patient_metadata = pd.read_excel(
    "data/integratedatlas_supplemental_files/SuppTable1.xlsx"
)

patient_metadata.head()

,donor_id,Basal,Her2,LumA,LumB,Normal
0,BC17086-12-Tumor,0.254329,0.429654,0.0,0.316017,0.0
1,BC17086-24-Tumor,0.479841,0.333790,0.0,0.186369,0.0
2,BC17086-25-Tumor,0.044725,0.482418,0.0,0.472856,0.0
3,BC17086-35-Tumor,0.000000,0.567416,0.0,0.432584,0.0
4,BC258-Tumor,0.000000,0.487568,0.0,0.512432,0.0


In [4]:
print(patient_metadata.shape)
patient_metadata.columns.tolist()

(138, 6)


['donor_id', 'Basal', 'Her2', 'LumA', 'LumB', 'Normal']

In [5]:
patient_metadata.isna().sum().sort_values(ascending=False)

donor_id    0
Basal       0
Her2        0
LumA        0
LumB        0
Normal      0
dtype: int64

In [6]:
patient_metadata.head(20)

,donor_id,Basal,Her2,LumA,LumB,Normal
0,BC17086-12-Tumor,0.254329,0.429654,0.000000,0.316017,0.000000
1,BC17086-24-Tumor,0.479841,0.333790,0.000000,0.186369,0.000000
2,BC17086-25-Tumor,0.044725,0.482418,0.000000,0.472856,0.000000
3,BC17086-35-Tumor,0.000000,0.567416,0.000000,0.432584,0.000000
4,BC258-Tumor,0.000000,0.487568,0.000000,0.512432,0.000000
5,BC302-Tumor,0.000000,0.341736,0.000000,0.658264,0.000000
6,BC389-Tumor,0.000000,0.000000,0.708314,0.117440,0.174246
7,BC392-Tumor,0.000000,0.000000,0.581742,0.000000,0.418258
8,BC393-Tumor,0.000000,0.000000,0.555948,0.444052,0.000000
9,BC394-Tumor,0.075120,0.493940,0.000000,0.430940,0.000000


In [8]:
# Predicted subtype for each patient
patient_metadata["Predicted_Subtype"] = patient_metadata[
    ["Basal", "Her2", "LumA", "LumB", "Normal"]
].idxmax(axis=1)

patient_metadata.head()

,donor_id,Basal,Her2,LumA,LumB,Normal,Predicted_Subtype
0,BC17086-12-Tumor,0.254329,0.429654,0.0,0.316017,0.0,Her2
1,BC17086-24-Tumor,0.479841,0.333790,0.0,0.186369,0.0,Basal
2,BC17086-25-Tumor,0.044725,0.482418,0.0,0.472856,0.0,Her2
3,BC17086-35-Tumor,0.000000,0.567416,0.0,0.432584,0.0,Her2
4,BC258-Tumor,0.000000,0.487568,0.0,0.512432,0.0,LumB


In [9]:
patient_metadata["Predicted_Subtype"].value_counts()


Predicted_Subtype
LumB      40
Basal     37
LumA      29
Her2      21
Normal    11
Name: count, dtype: int64

In [10]:
for f in [
    "SuppTable2 (epi).xlsx",
    "SuppTable3 (imm).xlsx",
    "SuppTable4 (strom).xlsx",
]:
    df = pd.read_excel(f"data/integratedatlas_supplemental_files/{f}")
    print(f)
    print(df.shape)
    print(df.columns.tolist()[:20])
    print("-"*80)

SuppTable2 (epi).xlsx
(100, 7)
['gene', 'coef', 'mean', 'pval', 'fdr', 'group', 'cluster']
--------------------------------------------------------------------------------
SuppTable3 (imm).xlsx
(48, 6)
['cluster', 'cell_type', 'sub_type', 'Cell Ontology', 'notes', 'action']
--------------------------------------------------------------------------------
SuppTable4 (strom).xlsx
(17, 6)
['cluster', 'cell_type', 'sub_type', 'Cluster', 'cell ontology', 'notes']
--------------------------------------------------------------------------------
